In [1]:
from appworld import AppWorld, load_task_ids
import json
import os
from datetime import datetime
from pathlib import Path
from typing import Dict
from tqdm import tqdm
from pydantic import BaseModel, Field
from typing import List, Literal, Optional, Tuple
from openai import OpenAI

In [2]:
client = OpenAI(
    base_url="https://3f4bdsdpetv6x5-8000.proxy.runpod.net/v1",
    api_key="dummy",  # or your real key if you used --api-key
)

In [3]:
resp = client.chat.completions.create(
    model="microsoft/Phi-3-mini-4k-instruct",
    messages=[
        {"role": "user", "content": "Explain gravity in one sentence."}
    ],
    max_tokens=50,
    logprobs=True,          # <-- REQUIRED
)

In [4]:
from baseline.templates import Template
from baseline.config import Config

# models.py
from pydantic import BaseModel
from typing import List, Literal


class Message(BaseModel):
    """Single conversation message"""
    role: Literal["system", "user", "assistant"]
    content: str
    log_probs: Optional[List[Tuple[str, float]]] = None


class AgentState(BaseModel):
    conversation_history: List[Message] = Field(default_factory=list)
    log_probs: List[float] = Field(default_factory=list)
    iteration: int = 0
    done: bool = False
    max_iters: int = 50
    
    @property 
    def should_continue(self):
        return self.iteration < self.max_iters and not self.done
    
    def total_chars(self):
        return sum(len(msg.content) for msg in self.conversation_history)

In [5]:
import re

def message_parser_with_position(message: str) -> Tuple[Optional[str], Optional[int], Optional[int]]:
    """
    Extract code from markdown code blocks and return code + positions.
    Returns (code, start_pos, end_pos) where positions mark the full code block including ```.
    """
    pattern = r'```(?:python)?\n(.*?)```'
    match = re.search(pattern, message, re.DOTALL)
    
    if match:
        code = match.group(1).strip()
        # match.start() is the position of the opening ```
        # match.end() is the position after the closing ```
        return code, match.start(), match.end()
    
    # Fallback for non-markdown code
    if message.strip().startswith(('print(', 'apis.', 'import ', 'from ')):
        return message.strip(), 0, len(message)
    
    return None, None, None

def truncate_message_history(
    conversation_history: List[Message], 
    threshold: int
) -> List[Message]:
    """
    Truncate conversation history if it exceeds the character threshold.
    Keeps the system message and most recent messages.
    """
    total_chars = sum(len(msg.content) for msg in conversation_history)
    
    if total_chars <= threshold:
        return conversation_history
    
    # Always keep the first message (initial prompt with instructions)
    truncated = [conversation_history[0]]
    
    # Keep most recent messages until we're under threshold
    recent_messages = []
    current_chars = len(conversation_history[0].content)
    
    # Work backwards from most recent
    for msg in reversed(conversation_history[1:]):
        msg_chars = len(msg.content)
        if current_chars + msg_chars <= threshold:
            recent_messages.insert(0, msg)
            current_chars += msg_chars
        else:
            break
    
    truncated.extend(recent_messages)
    return truncated

In [6]:
from dataclasses import dataclass
import os
from dotenv import load_dotenv

load_dotenv()

@dataclass 
class Config:
    # Agent parameters
    max_iters: int = 50  # Match the paper's baseline
    
    # OpenAI parameters
    openai_api_key: str = os.getenv("OPENAI_API_KEY")
    service: str = "vLLM" # can set to OpenAI or TogetherAI
    togetherai_api_key: str = os.getenv("TOGETHER_AI")
    base_model: str = os.getenv("VLLM_MODEL") 
    max_tokens: int = 512
    temperature: float = 0.0
    
    # Context management
    truncation_threshold: int = 12000  # Characters, not tokens

    # AppWorld Root
    os.environ["APPWORLD_ROOT"] = os.getenv("APPWORLD_ROOT")
    
    @classmethod
    def for_model(cls, model_name: str):
        """Factory method for different model configs"""
        configs = {
            "gpt-4o": cls(base_model="gpt-4o-2024-05-13"),
            "gpt-4": cls(base_model="gpt-4-turbo-2024-04-09"),
            "o1": cls(
                base_model="o1-preview-2024-09-12",
                temperature=1.0,  # o1 requires temperature=1
                max_tokens=4000
            )
        }
        return configs.get(model_name, cls())

config = Config()

In [7]:
max_iters: int = config.max_iters
max_tokens: int = config.max_tokens
temperature: float = config.temperature
base_model: str = config.base_model
truncation_threshold: int = config.truncation_threshold
template = Template()
state = AgentState(max_iters=config.max_iters)
seed = None

In [8]:
task_ids = load_task_ids("train") # loads train ids, other options: dev, test_normal, test_challenge
task_id = task_ids[0]
world = AppWorld(task_id=task_id)

In [9]:
first_name = "John"
last_name = "Smith"
email = "john@yahoo.com"
phone_number = "1234567894"

init_template = template.format_prompt(
    first_name, 
    last_name, 
    email, 
    phone_number, 
    world.task.instruction
)

In [10]:
K=6
sets=["train"]

In [11]:
train_ids = [
		    tid
		    for dataset_name in sets
		    for tid in load_task_ids(dataset_name)
		]

In [12]:
task_id = train_ids[1]

In [13]:
import uuid
random_uuid = uuid.uuid4()

In [14]:
task_result = {
					"task_id": task_id,
					"completed": False,
					"iterations": 0,
					"error": None,
					"result": None,
					"conversation_length": 0,
					"token_log_probs": None,
					"unit_tests": None,
					"overall_success": None,
					"uuid": random_uuid
				}

In [15]:
from typing import Union
class ReactAgent:
    def __init__(self, config: Config, return_log_probs: bool = False, seed: int = None) -> None:
        self.max_iters: int = config.max_iters  # Fixed: use instance
        self.max_tokens: int = 512
        self.temperature: float = config.temperature
        self.base_model: str = config.base_model
        self.truncation_threshold: int = config.truncation_threshold
        self.template = Template()
        self.state = AgentState(max_iters=config.max_iters)
        self.seed = seed
        if config.service == "OpenAI":
            self.client = OpenAI(api_key=config.openai_api_key)
        elif config.service == "TogetherAI":
            os.environ["TOGETHER_API_KEY"] = config.togetherai_api_key
            self.client = OpenAI(
                api_key=config.togetherai_api_key,
                base_url="https://api.together.xyz/v1"
            )
        elif config.service == "vLLM":
            openai_api_key = "EMPTY"
            openai_api_base = "https://3f4bdsdpetv6x5-8000.proxy.runpod.net/v1"
            self.client = OpenAI(
                api_key=openai_api_key,
                base_url=openai_api_base,
            )
        self.eval_tracker: Dict = {}  

        
    def initialize(
        self, 
        first_name: str, 
        last_name: str, 
        email: str, 
        phone_number: str, 
        task_instructions: str
    ) -> None:
        init_template = self.template.format_prompt(
            first_name, 
            last_name, 
            email, 
            phone_number, 
            task_instructions
        )
        self.state.conversation_history.append(
            Message(role="user", content=init_template)
        )
    
    def call_llm(self, return_log_probs: bool = True) -> Tuple[str, Union[None, List[Tuple]]]: 
        messages = truncate_message_history(
            self.state.conversation_history, 
            self.truncation_threshold
        )

        extra_args = {}
        if self.seed:
            extra_args["seed"] = self.seed
        if return_log_probs:
            extra_args["logprobs"] = True
            extra_args["top_logprobs"] = 1  # only need the generated token

        response = self.client.chat.completions.create(
            model=self.base_model,
            messages=[msg.dict() for msg in messages],
            temperature=self.temperature,
            max_tokens=self.max_tokens,
            **extra_args,
        )

        choice = response.choices[0]
        text = choice.message.content

        if not return_log_probs:
            return text, None

        token_logprobs = []
        if choice.logprobs is not None:
            for item in choice.logprobs.content:
                token_logprobs.append((item.token, item.logprob))

        return text, token_logprobs
    
    def step(self, world):  
        llm_output, token_logprobs = self.call_llm(return_log_probs=True)
        
        # Log full output for analysis
        self.eval_tracker[f"iter_{self.state.iteration}_full_output"] = llm_output
        
        # Extract code and find its position
        code, code_start, code_end = message_parser_with_position(llm_output)
        observation_string = "No code block found in response."
        
        if code:
            try:
                observation = world.execute(code)
                observation_string = str(observation)
            except Exception as e:
                observation_string = f"Error: {str(e)}"
            
            truncated_logprobs = []
            found_opening = False
            backtick_count = 0
            
            for i, (token, logprob) in enumerate(token_logprobs):
                truncated_logprobs.append((token, logprob))
                
                # Count backtick tokens
                if token == '```':
                    backtick_count += 1
                    if backtick_count == 1:
                        found_opening = True
                    elif backtick_count == 2 and found_opening:
                        # Found closing backticks - stop here
                        break
            
            self.state.conversation_history.append(
                Message(role="assistant", content=llm_output[:code_end].strip(), log_probs=truncated_logprobs)
            )
        else:
            # No code found - store full response
            self.state.conversation_history.append(
                Message(role="assistant", content=llm_output, log_probs=token_logprobs)
            )
        
        # Append real observation
        self.state.conversation_history.append(
            Message(role="user", content=f"Output:\n```\n{observation_string}\n```")
        )
        
        self.state.iteration += 1
        
        if world.task_completed():
            self.state.done = True
        
        return world
    
    def run(self, world):
        """
        Execute agent loop until completion or max iterations
        """
        while self.state.should_continue:
            world = self.step(world)
            
            if self.state.done:
                print(f"Task completed in {self.state.iteration} iterations")
                break
        
        if not self.state.done:
            print(f"Max iterations ({self.max_iters}) reached without completion")
        
        return world


In [19]:
agent = ReactAgent(config)

In [20]:
agent.initialize(
                first_name,
                last_name,
                email,
                phone_number,
                world.task.instruction
            )	

In [22]:
agent.run(world)

Max iterations (50) reached without completion


In [23]:
print(agent.state.conversation_history[0].content)

USER:
    I am your supervisor and you are a super intelligent AI Assistant whose job is to achieve my day-to-day tasks completely autonomously.

    To do this, you will need to interact with app/s (e.g., spotify, venmo etc) using their associated APIs on my behalf. 
    For this you will undertake a *multi-step conversation* using a python REPL environment. That is, you will write the 
    python code and the environment will execute it and show you the result, based on which, you will write python code 
    for the next step and so on, until you've achieved the goal. This environment will let you interact with app/s using their associated APIs on my behalf.

    Here are three key APIs that you need to know to get more information

    # To get a list of apps that are available to you.
    print(apis.api_docs.show_app_descriptions())

    # To get the list of apis under any app listed above, e.g. spotify
    print(apis.api_docs.show_api_descriptions(app_name='spotify'))

    # To ge

In [24]:
print(agent.state.conversation_history[1].content)

Okay. Let's start by finding which APIs are available to use in Spotify.

Code:
```python
print(apis.api_docs.show_api_descriptions(app_name='spotify'))
```


In [25]:
agent.state.conversation_history[1].log_probs

[('Okay', -0.8606755137443542),
 ('.', -0.16079460084438324),
 ('Let', -0.6892639398574829),
 ("'", -0.00708164693787694),
 ('s', -1.2397689715726301e-05),
 ('start', -0.7037070393562317),
 ('by', -0.1041117012500763),
 ('finding', -0.2213335633277893),
 ('which', -0.4912150204181671),
 ('APIs', -0.0244450680911541),
 ('are', -0.014172264374792576),
 ('available', -0.0005096090608276427),
 ('to', -0.11178640276193619),
 ('use', -0.00048494499060325325),
 ('in', -0.018420346081256866),
 ('Sp', -0.016396544873714447),
 ('ot', -6.794906312279636e-06),
 ('ify', -1.2755313036905136e-05),
 ('.', -0.13459846377372742),
 ('\n', -0.021749665960669518),
 ('\n', -0.7597009539604187),
 ('Code', -0.09781382232904434),
 (':', -0.0010702840518206358),
 ('\n', -0.0006748544401489198),
 ('```', -0.001720973290503025),
 ('python', -0.00014602071314584464),
 ('\n', -0.00025340684805996716),
 ('print', -0.004201632924377918),
 ('(', -2.109982233378105e-05),
 ('apis', -1.883488948806189e-05),
 ('.', -6.556

In [26]:
count = 0

for token, prob in agent.state.conversation_history[1].log_probs:
    count += len(token)

count

144

In [27]:
extra_args = {}

extra_args["logprobs"] = True
extra_args["top_logprobs"] = 1  

response = agent.client.chat.completions.create(
    model=agent.base_model,
    messages=[{"role": "user", "content": str(agent.state.conversation_history[0].content)}],
    temperature=agent.temperature,
    max_tokens=agent.max_tokens,
    **extra_args
)

In [28]:
choice = response.choices[0]
text = choice.message.content

token_logprobs = []
if choice.logprobs is not None:
    for item in choice.logprobs.content:
        token_logprobs.append((item.token, item.logprob))

In [29]:
# Extract code and find its position
llm_output = text

code, code_start, code_end = message_parser_with_position(llm_output)
observation_string = "No code block found in response."

In [30]:
code_end

150

In [31]:
# Truncate to include everything up to and including the code block
truncated_response = llm_output[:code_end].strip()

In [32]:
truncated_response

"Okay. Let's first find which APIs are available to use in Spotify.\nCode:\n```python\nprint(apis.api_docs.show_api_descriptions(app_name='spotify'))\n```"

In [33]:
# Build up tokens until exact match

truncated_logprobs = []
found_opening = False
backtick_count = 0

for i, (token, logprob) in enumerate(token_logprobs):
    truncated_logprobs.append((token, logprob))
    
    # Count backtick tokens
    if token == '```':
        backtick_count += 1
        if backtick_count == 1:
            found_opening = True
        elif backtick_count == 2 and found_opening:
            # Found closing backticks - stop here
            break


In [34]:
truncated_logprobs

[('Okay', -0.8550538420677185),
 ('.', -0.6943853497505188),
 ('Let', -0.5850694179534912),
 ("'", -0.004117345437407494),
 ('s', -3.576278118089249e-07),
 ('first', -0.6991567611694336),
 ('find', -0.009626161307096481),
 ('which', -0.08534575998783112),
 ('APIs', -0.003921795636415482),
 ('are', -0.004923244938254356),
 ('available', -0.0004058252670802176),
 ('to', -0.011996148154139519),
 ('use', -0.0013469918631017208),
 ('in', -0.006555125582963228),
 ('Sp', -0.010243344120681286),
 ('ot', -7.629365427419543e-06),
 ('ify', -4.768370445162873e-07),
 ('.', -0.02847280167043209),
 ('\n', -0.005618968512862921),
 ('Code', -0.07161795347929001),
 (':', -0.0003045333724003285),
 ('\n', -0.00012611546844709665),
 ('```', -0.0005515484372153878),
 ('python', -0.00032610344351269305),
 ('\n', -4.8397800128441304e-05),
 ('print', -0.0009924016194418073),
 ('(', -1.1920858014491387e-05),
 ('apis', -4.172316494077677e-06),
 ('.', -2.3841830625315197e-06),
 ('api', -4.410646579344757e-05),
 (

In [35]:
for tup in truncated_logprobs:
    print(repr(tup[0]))

'Okay'
'.'
'Let'
"'"
's'
'first'
'find'
'which'
'APIs'
'are'
'available'
'to'
'use'
'in'
'Sp'
'ot'
'ify'
'.'
'\n'
'Code'
':'
'\n'
'```'
'python'
'\n'
'print'
'('
'apis'
'.'
'api'
'_'
'docs'
'.'
'show'
'_'
'api'
'_'
'des'
'cri'
'ptions'
'('
'app'
'_'
'name'
"='"
'spot'
'ify'
"'))"
'\n'
'```'


In [36]:
task_result["completed"] = world.task_completed()

In [37]:
task_result["iterations"] = agent.state.iteration

In [38]:
task_result["conversation_length"] = len(agent.state.conversation_history)

In [39]:
task_result["token_log_probs"] = [
						    msg.log_probs
						    for msg in agent.state.conversation_history
						    if msg.role == "assistant"
						]

In [40]:
task_result["assistant_messages"] = [
						    msg.content
						    for msg in agent.state.conversation_history
						    if msg.role == "assistant"
						]

In [41]:
world.safety_guard.disable()

results = world.evaluate()

────────────────────────────────────────────────── Overall Stats ──────────────────────────────────────────────────

Num Passed Tests : 1

Num Failed Tests : 1

Num Total  Tests : 2

───────────────────────────────────────────────────── Passes ──────────────────────────────────────────────────────

...

assert no model changes.

────────────────────────────────────────────────────── Fails ──────────────────────────────────────────────────────

>> Failed Requirement

assert answers match.

```python
with test(
    """
    assert answers match.
    """
):
    test.answer(predicted_answer, ground_truth_answer)
```
----------
AssertionError:  '<<not_given>>' == 'a love that never was'

In [42]:
type(results)

appworld.evaluator.TestTracker

In [43]:
results.pass_count

1

In [44]:
results.fail_count

1

In [45]:
outs = results.to_dict()

In [46]:
outs.keys()

dict_keys(['success', 'difficulty', 'num_tests', 'passes', 'failures'])

In [47]:
outs["num_tests"]

2

In [48]:
len(outs["passes"])

1

In [50]:
task_result["overall_success"] = len(evaluation['passes'])/evaluation['num_tests']

NameError: name 'evaluation' is not defined

In [ ]:
task_result["evaluation_details"] = evaluation

In [ ]:
task_result

In [ ]:
task_result["agent_state"] = agent.state

In [ ]:
task_result["agent_state"]

In [51]:
for message in range(len(task_result["agent_state"].conversation_history)):
    text = task_result["agent_state"].conversation_history[message]
    print("\n \n")
    print(f"{text.role}: {text.content}")

KeyError: 'agent_state'

In [ ]:
task_set = task_id

In [ ]:
all_rollouts = []
all_rollouts.append(task_result)

In [ ]:
all_rollouts[0]["task_id"]

In [ ]:
rollout = all_rollouts[0]

In [ ]:
rollout_reward = rollout["overall_success"]

In [ ]:
rollout_id = rollout["uuid"]

In [ ]:
baseline_reward = 1

In [ ]:
LOO_advantage = rollout_reward - baseline_reward

In [ ]:
rollout["advantage"] = LOO_advantage

In [ ]:
updated_rollouts = []

In [ ]:
updated_rollouts.append(rollout)

In [ ]:
updated_rollouts

In [ ]:
rollout.keys()

In [ ]:
messages = rollout["agent_state"].conversation_history
assistant_messages = rollout["assistant_messages"]
token_log_probs_list = rollout["token_log_probs"]

In [ ]:
messages[3].role

In [ ]:
assistant_messages[1]

In [ ]:
context = messages[:3]

In [ ]:
context

In [ ]:
context_text = [x.content for x in context]

In [ ]:
full_text = ' '.join(context_text)

In [ ]:
type(full_text)

In [ ]:
for x in context:
    print(x.role)

In [ ]:
!pip install transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

In [ ]:
!pip install torch

In [ ]:
import torch
full_ids = tokenizer.encode(' '.join(context_text), return_tensors="pt")

In [ ]:
rollout["agent_state"].conversation_history[1].content

In [ ]:
rollout["agent_state"].conversation_history[1].log_probs

In [ ]:
rollout["agent_state"].conversation_history[3].role

In [ ]:
context_msgs = messages[:3]

In [ ]:
context_str = "".join(m.content for m in context_msgs)

In [ ]:
print(context_str)

In [ ]:
msg = rollout["agent_state"].conversation_history[3]

In [ ]:
assistant_text = msg.content

In [ ]:
full_str = context_str + assistant_text

In [ ]:
print(full_str)

In [ ]:
token_log_probs_list=rollout["token_log_probs"]

In [ ]:
old_token_log_probs = token_log_probs_list[0]

In [ ]:
A = len(old_token_log_probs)

In [ ]:
A

In [ ]:
T = full_ids.shape[1]

In [52]:
agent.state.conversation_history)

list

In [62]:
!pip install torch --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu


  Obtaining dependency information for torch from https://download.pytorch.org/whl/cpu/torch-2.9.1%2Bcpu-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 110.9/110.9 MB 9.5 MB/s eta 0:00:00
Using cached https://download.pytorch.org/whl/cpu/torch-2.9.1%2Bcpu-cp311-cp311-win_amd64.whl (110.9 MB)


In [63]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-128k-instruct",
    trust_remote_code=True,
)

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\Aaron McClendon\Documents\github\maml-agent\env311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.